In [19]:
# ================= Cell 1: 基础设置与核心函数 =================

# --- 建模逻辑说明 ---
# 本项目采用复杂网络理论中的 L-Space（空间L）建模方法。
# 定义：将每一个地铁站抽象为“节点”，将车站之间直接相连的物理轨道抽象为“边”。
# 选择理由：这种建模方式能够最直观地还原地铁系统的物理拓扑结构，是分析网络连通性、
# 物理路径长度以及评估系统在面对随机故障或目标攻击时（鲁棒性分析）的基础。

%matplotlib inline

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import os
import glob
from matplotlib import cm
import numpy as np
import warnings

# 屏蔽非致命性的警告信息，以保持实验环境的整洁和分析结果的聚焦
warnings.filterwarnings('ignore')

# -----------------------------------------------------------
# 1. 环境配置
# -----------------------------------------------------------
# 数据存放路径
DATA_PATH = r'数据爬取\高德API\整合数据（含环和分叉）'

# 中文字体配置：确保图表中的中文城市名称与站名能够正常渲染
# 注意：SimHei 适用于大部分 Windows 环境
plt.rcParams['font.sans-serif'] = ['SimHei'] 
plt.rcParams['axes.unicode_minus'] = False

# -----------------------------------------------------------
# 2. 数据预处理函数
# -----------------------------------------------------------

def get_city_list(path):
    """
    【功能】：遍历数据文件夹，自动提取所有已整理完成的城市名称。
    【设计优势】：增强了代码的可扩展性，当文件夹中新增城市数据时，无需修改代码即可自动识别。
    """
    if not os.path.exists(path):
        print(f"路径不存在：{path}")
        return []
    
    files = glob.glob(os.path.join(path, "*_All_Stations.csv"))
    cities = []
    for f in files:
        filename = os.path.basename(f)
        city_name = filename.split('_All_Stations')[0]
        cities.append(city_name)
    return sorted(list(set(cities)))

def load_data(city_name):
    """
    【功能】：从 CSV 原始文件中读取数据，执行数据清洗，并构建 NetworkX 拓扑图模型。
    【关键处理】：
    1. 自环过滤：自动删除起点和终点 ID 相同的错误区间数据，防止逻辑冗余。
    2. 空间位置关联：将经纬度坐标作为节点属性，为后续的地理拓扑绘图提供坐标支持。
    """
    station_file = os.path.join(DATA_PATH, f"{city_name}_All_Stations.csv")
    edge_file = os.path.join(DATA_PATH, f"{city_name}_All_Edges.csv")
    
    if not os.path.exists(station_file) or not os.path.exists(edge_file):
        print(f"数据文件缺失：无法加载 {city_name} 的站点或区间数据")
        return None, None

    try:
        stations_df = pd.read_csv(station_file, encoding='utf-8-sig')
        edges_df = pd.read_csv(edge_file, encoding='utf-8-sig')
        
        # 【数据清洗】删除所有逻辑错误的自环（起点 ID 等于 终点 ID）
        edges_df = edges_df[edges_df['from_id'] != edges_df['to_id']] 
        
        if stations_df.empty or edges_df.empty:
            return None, None

        # 初始化无向图模型
        G = nx.Graph()
        
        # 建立 站点ID 到 物理位置(经纬度) 和 站名 的字典映射
        # 通过 groupby 取 mean() 是为了消除同一个 ID 在不同记录中可能存在的微小坐标扰动
        nodes_info = stations_df.groupby('station ID').agg({
            '站名': 'first', '经度': 'mean', '纬度': 'mean'
        }).to_dict('index')

        # 向图中注入车站节点及地理位置属性
        for s_id, info in nodes_info.items():
            G.add_node(s_id, pos=(info['经度'], info['纬度']), label=info['站名'])

        # 向图中注入轨道边属性，关联对应的地铁线路名称
        for _, row in edges_df.iterrows():
            if row['from_id'] in G and row['to_id'] in G:
                G.add_edge(row['from_id'], row['to_id'], line=row['线路'])
        
        return G, edges_df

    except Exception as e:
        print(f"读取城市数据 {city_name} 时发生错误：{e}")
        return None, None

# -----------------------------------------------------------
# 3. 核心分析与可视化函数
# -----------------------------------------------------------

def visualize_city(city_name):
    """
    【功能】：计算选定城市地铁网络的关键指标，打印分析报告并生成地理拓扑可视化图表。
    """
    G, edges_df = load_data(city_name)
    if G is None: return

    # --- A. 指标计算与名词解释 ---
    num_nodes = G.number_of_nodes() # 车站总数
    num_edges = G.number_of_edges() # 轨道区间总数
    
    # 1. 平均度 (Average Degree)：描述网络中车站的平均连接能力。
    # 【解释】：平均每个站向几个方向延伸。数值越高，意味着网络的整体换乘便利性越好。
    degrees = dict(G.degree())
    avg_degree = sum(degrees.values()) / num_nodes
    
    # 2. 网络密度 (Density)：衡量地铁网的密集程度。
    # 【解释】：反映该城市地铁网已经成“网”的程度。数值越大，表示站与站之间的连接越紧密。
    density = nx.density(G)
    
    # 3. 度中心性 (Degree Centrality)：识别网络中的核心换乘枢纽。
    # 【物理意义】：多线换乘大站的该得分极高，它们是维持整个网络运行效率的关键节点。
    degree_cent = nx.degree_centrality(G)
    top_degree = sorted(degree_cent.items(), key=lambda x: x[1], reverse=True)[:5]
    node_labels = nx.get_node_attributes(G, 'label')

    # --- B. 打印文字报告 ---
    print("1. 平均度 (Average Degree)：描述网络中车站的平均连接能力。")
    print("[专业表述]：衡量网络整体的物理连通性基数。")
    print("[物理意义]:平均每个站向几个方向延伸。数值越高，意味着网络的整体换乘便利性越好。反映了轨道交通系统在物理层面的换乘冗余度与综合连通性能。")
    print("2. 网络密度 (Density)：衡量地铁网的密集程度。")
    print("[专业表述]：量化线网的拓扑紧凑程度与资源分配效率。")
    print("[物理意义]: 反映该城市地铁网已经成“网”的程度。数值越大，表示站与站之间的连接越紧密。")
    print("          在轨道交通网中，密度常用于评估线网从“线性放射状”向“成熟网格状”演化的程度，是衡量线网覆盖深度与拓扑稳健性的核心宏观指标。")
    print("3. 度中心性 (Degree Centrality)：识别网络中的核心换乘枢纽。")
    print("[专业表述]：识别线网中的物理拓扑枢纽 (Topological Hubs)。")
    print("[物理意义]：度中心性量化了车站作为物理汇聚点的能力。多线换乘大站的该得分极高，它们是维持整个网络运行效率的关键节点。")
    print(f"\n--- {city_name} 地铁网络拓扑指标分析报告 ---")
    print(f"[网络规模] 车站总数: {num_nodes} | 轨道区间总数: {num_edges}")
    print(f"[连通水平] 平均连接度: {avg_degree:.2f} (反映平均换乘选择数量)")
    print(f"[网络密度] {density:.6f} (数值越高，路网结构越紧密)")
    print(f"[关键枢纽节点 Top 5] (换乘权重最高的车站):")
    for i, (node_id, val) in enumerate(top_degree):
        name = node_labels.get(node_id, str(node_id))
        print(f"    第 {i+1} 名: {name} (中心度得分: {val:.4f})")

    # --- C. 绘图：地理空间还原 ---
    plt.figure(figsize=(12, 10))
    ax = plt.gca()
    pos = nx.get_node_attributes(G, 'pos')
    
    # 获取线路唯一列表并映射颜色。使用 tab20 调色盘以确保多线路区分明显。
    unique_lines = edges_df['线路'].unique()
    colors = cm.get_cmap('tab20', len(unique_lines))
    line_color_map = {line: colors(i % 20) for i, line in enumerate(unique_lines)}

    # 1. 分线路绘制轨道（边）
    for line in unique_lines:
        line_data = edges_df[edges_df['线路'] == line]
        edge_list = [(u, v) for u, v in zip(line_data['from_id'], line_data['to_id']) if G.has_edge(u, v)]
        if edge_list:
            nx.draw_networkx_edges(G, pos, edgelist=edge_list, 
                                 edge_color=[line_color_map[line]], width=3, alpha=0.8, ax=ax)

    # 2. 绘制车站（节点）
    nx.draw_networkx_nodes(G, pos, node_size=20, node_color='black', alpha=0.5, ax=ax)
    
    # 3. 动态构建图例：将图例置于绘图区外侧，防止遮挡地理拓扑结构
    legend_elements = [plt.Line2D([0], [0], color=line_color_map[l], lw=4, label=l) for l in unique_lines]
    ax.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.01, 1), 
              title="线路图例", fontsize=9, frameon=False)

    plt.title(f"{city_name} 地铁网络地理拓扑可视化 (L-Space 模型)", fontsize=18, pad=20)
    
    # 【关键选择】：设置坐标轴比例相等，确保经纬度不缩放变形，从而真实反映城市轮廓
    plt.axis('equal') 
    plt.grid(True, linestyle=':', alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(" --- 建模逻辑说明 ---")
    print(" 本项目采用复杂网络理论中的 L-Space（空间L）建模方法。")
    print(" 定义：将每一个地铁站抽象为“节点”，将车站之间直接相连的物理轨道抽象为“边”。")
    print(" 选择理由：这种建模方式能够最直观地还原地铁系统的物理拓扑结构，是分析网络连通性、")
    print("物理路径长度以及评估系统在面对随机故障或目标攻击时（鲁棒性分析）的基础。")

print("基础环境配置与核心分析函数已加载完毕。")

# ================= Cell 2: 方案一 (交互式下拉菜单) =================

import ipywidgets as widgets

# 获取当前数据集包含的城市列表
city_options = get_city_list(DATA_PATH)

if not city_options:
    print("未检测到有效数据：请检查 Cell 1 中的数据路径配置。")
else:
    print(f"数据加载成功：共检索到 {len(city_options)} 个城市的地铁网络数据。")

    # 创建交互式下拉组件
    dropdown = widgets.Dropdown(
        options=city_options,
        value=city_options[0],
        description='选择城市:',
        style={'description_width': 'initial'}
    )

    # 绑定交互功能：选择不同城市时将实时触发指标分析与图形绘制
    widgets.interact(visualize_city, city_name=dropdown);

基础环境配置与核心分析函数已加载完毕。
数据加载成功：共检索到 48 个城市的地铁网络数据。


interactive(children=(Dropdown(description='选择城市:', options=('Beijing', 'Changchun', 'Changsha', 'Changzhou', …

In [20]:
# ================= Cell 1: 核心设置与高级分析函数 =================

# --- 逻辑说明 ---
# 本代码块旨在通过更精细的维度观察地铁网络。
# 1. 节点分级：在 L-Space 模型中，度（Degree）代表一个站连接的相邻车站数。
#    - 度 > 2 的站点通常是三岔路口或多线换乘站，是网络的“咽喉”。
#    - 我们通过视觉上的大小区分，让观察者一眼识别出城市的交通枢纽。
# 2. 鲁棒性基础：计算连通分量和直径。
#    - 如果一个地铁网是不连通的，意味着部分线路之间无法通过地铁直接到达，这对路径规划有重大影响。

def visualize_city_advanced(city_name):
    """
    高级分析与绘图函数
    功能：实现节点分类显示、线路着色可视化，并提供网络效率（直径/路径）深度报告。
    """
    G, edges_df = load_data(city_name)
    
    if G is None or G.number_of_nodes() == 0:
        print(f"数据加载失败：{city_name} 的数据无效或为空。")
        return

    # --- 1. 节点层级筛选逻辑 ---
    # 定义：度（Degree）大于 2 的节点。
    # 理由：在普通的线性线路中，中间站的度为 2（一前一后）。
    # 当度 > 2 时，说明该站是一个分叉点或换乘点，是网络中的“关键少数”。
    transfer_stations = [n for n, d in G.degree() if d > 2]
    normal_stations = [n for n in G.nodes() if n not in transfer_stations]
    
    pos = nx.get_node_attributes(G, 'pos')
    
    print("1. 网络直径（Diameter）")
    print("[专业表述]：量化网络拓扑结构的跨度极限。")
    print("[物理意义]: 代表网络中最远两个节点间的最小测地距离（Geodesic Distance）。代表了该城市地铁通行的“最差情况”。")
    print("          它界定了城市轨道交通系统在最不利路径下的通达上限，反映了网络在覆盖广域地理空间时的拓扑扩张成本。")
    print("2. 平均路径（Average Path Length）")
    print("[专业表述]：衡量系统全局的静态传输效率。")
    print("[物理意义]: 反映了全网随机站点对之间通勤的平均空间开销。随机选两站，平均需要经过多少站。代表了整体出行效率。")
    print("          该值越小，代表系统在拓扑层面的平均通勤路径越短，体现了线网设计在缩短乘客时耗上的整体效能。")

    # --- 2. 绘图与视觉增强 ---
    plt.figure(figsize=(12, 10))
    ax = plt.gca()

    # (A) 绘制轨道（边）：按线路分配颜色
    unique_lines = edges_df['线路'].unique()
    # 使用 tab20 调色盘：该色盘提供 20 种高辨识度颜色，适合地铁多线路区分
    colors = cm.get_cmap('tab20', len(unique_lines))
    line_color_map = {line: colors(i % 20) for i, line in enumerate(unique_lines)}

    for line in unique_lines:
        line_data = edges_df[edges_df['线路'] == line]
        edge_list = list(zip(line_data['from_id'], line_data['to_id']))
        valid_edges = [(u, v) for u, v in edge_list if G.has_edge(u, v)]
        if valid_edges:
            nx.draw_networkx_edges(G, pos, edgelist=valid_edges, 
                                 edge_color=[line_color_map[line]], width=2.5, ax=ax)

    # (B) 分级绘制车站（节点）
    # 普通站点：较小，降低透明度，作为背景衬托
    nx.draw_networkx_nodes(G, pos, nodelist=normal_stations, 
                         node_size=10, node_color='black', alpha=0.4, ax=ax)
    # 枢纽站点：较大，完全不透明，突出其核心地位
    nx.draw_networkx_nodes(G, pos, nodelist=transfer_stations, 
                         node_size=20, node_color='black', alpha=1.0, ax=ax)

    # (C) 图例与装饰
    legend_elements = [plt.Line2D([0], [0], color=line_color_map[l], lw=3, label=l) for l in unique_lines]
    ax.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.01, 1), 
              title="地铁线路图例", fontsize=10, frameon=False)
    
    plt.title(f"{city_name} 地铁拓扑分析 (枢纽站分级标注)", fontsize=18, pad=20)
    plt.axis('equal') # 保持经纬度比例，防止城市形状被拉伸
    plt.grid(True, linestyle=':', alpha=0.3)
    plt.tight_layout()
    plt.show()

    # --- 3. 计算与专业报告输出 ---
    # 此部分输出用于量化评估地铁网络的运行效率和覆盖广度。
    
    print(f"\n--- {city_name} 地铁网络深度指标报告 ---")
    
    # 连通性分析（Connectivity Analysis）
    # [名词解释]：连通性代表网络中是否存在“孤岛”。如果不连通，乘客无法仅靠地铁到达所有站点。
    is_connected = nx.is_connected(G)
    components = nx.number_connected_components(G)
    num_nodes = G.number_of_nodes()
    num_edges = G.number_of_edges()
    
    print(f"[基础规模] 车站数: {num_nodes} | 轨道段数: {num_edges}")
    print(f"[连通状态] 全网连通性: {'良好 (已全面覆盖)' if is_connected else '存在孤岛 (线路未完全连接)'}")
    print(f"[系统碎片] 独立连通区域数: {components} (数值为 1 表示全网通达)")

    # 网络直径（Diameter）与平均路径（Average Path Length）
    # [名词解释] 直径：网络中最远两站之间的最短距离。代表了该城市地铁通行的“最差情况”。
    # [名词解释] 平均最短路径：随机选两站，平均需要经过多少站。代表了整体出行效率。
    if is_connected:
        diameter = nx.diameter(G)
        avg_path = nx.average_shortest_path_length(G)
        print(f"[通行效率] 网络直径: {diameter} 站 (穿越城市最远两端的成本)")
        print(f"[平均水平] 平均出行站数: {avg_path:.2f} 站")
    else:
        # 统计学处理：对于不连通图，直径为无穷大，因此需取其“最大连通子图”（即最大的那个核心网）进行分析
        largest_cc_nodes = max(nx.connected_components(G), key=len)
        G_sub = G.subgraph(largest_cc_nodes)
        sub_diameter = nx.diameter(G_sub)
        sub_avg_path = nx.average_shortest_path_length(G_sub)
        print(f"[风险提示] 注意：由于网络不连通，以下指标基于最大核心网 (节点数: {len(largest_cc_nodes)}) 计算：")
        print(f"    - 核心网直径: {sub_diameter} 站")
        print(f"    - 核心网平均出行站数: {sub_avg_path:.2f} 站")

    print(" 本代码块旨在通过更精细的维度观察地铁网络。")
    print("1. 节点分级：在 L-Space 模型中，度（Degree）代表一个站连接的相邻车站数。")
    print("- 度 > 2 的站点通常是三岔路口或多线换乘站，是网络的“咽喉”。")
    print("- 我们通过视觉上的大小区分，让观察者一眼识别出城市的交通枢纽。")
    print("2. 鲁棒性基础：计算连通分量和直径。")
    print("- 如果一个地铁网是不连通的，意味着部分线路之间无法通过地铁直接到达，这对路径规划有重大影响。")

print("核心分析函数已加载完毕。")

# ================= Cell 2: 交互式下拉菜单方案 =================

import ipywidgets as widgets

# 调用 Cell 1 中定义的 get_city_list 获取数据目录下的所有城市
city_options = get_city_list(DATA_PATH)

if not city_options:
    print("路径下未发现城市数据，请检查 Cell 1 的 DATA_PATH 配置。")
else:
    print(f"系统就绪：已识别 {len(city_options)} 个城市的地铁模型。")
    
    dropdown = widgets.Dropdown(
        options=city_options,
        value=city_options[0],
        description='选择城市分析:',
        style={'description_width': 'initial'},
        disabled=False,
    )

    # 绑定交互，自动触发 visualize_city_advanced 函数
    widgets.interact(visualize_city_advanced, city_name=dropdown);

核心分析函数已加载完毕。
系统就绪：已识别 48 个城市的地铁模型。


interactive(children=(Dropdown(description='选择城市分析:', options=('Beijing', 'Changchun', 'Changsha', 'Changzhou'…

In [21]:
# ================= Cell 1: P-Space 核心设置与函数定义 =================

# --- 建模逻辑说明 ---
# P-Space（空间 P）模型与之前使用的 L-Space 模型有本质不同：
# 1. 定义：在 P-Space 中，只要两个车站属于同一条地铁线路，它们之间就会建立一条直接的“边”。
# 2. 物理意义：边不再代表一段物理轨道，而是代表“直达可能性”。
# 3. 路径意义：在 P-Space 中，路径长度为 1 代表直达，为 2 代表需换乘 1 次，以此类推。
# 4. 选择理由：这种建模方法能够直接反映地铁网络的“换乘便捷性”，是评价乘客心理距离的重要工具。

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import os
import glob
from itertools import combinations
from matplotlib import cm
import numpy as np
import warnings
import ipywidgets as widgets
from IPython.display import display

# 屏蔽不必要的兼容性警告
warnings.filterwarnings('ignore')

# -----------------------------------------------------------
# 1. 数据处理函数：构建 P-Space 拓扑图
# -----------------------------------------------------------

def get_city_list(path):
    """扫描文件夹以识别所有可用的城市地铁数据集"""
    if not os.path.exists(path):
        print(f"路径不存在: {path}")
        return []
    
    files = glob.glob(os.path.join(path, "*_All_Stations.csv"))
    cities = []
    for f in files:
        filename = os.path.basename(f)
        city_name = filename.split('_All_Stations')[0]
        cities.append(city_name)
    return sorted(list(set(cities)))

def load_pspace_data(city_name):
    """
    【核心逻辑】：构建 P-Space 完备子图（Clique）
    同一个线路上的所有站点进行“两两全连接”，模拟无需换乘即可到达的逻辑。
    """
    station_file = os.path.join(DATA_PATH, f"{city_name}_All_Stations.csv")
    
    if not os.path.exists(station_file):
        print(f"文件缺失: {city_name}")
        return None, None

    try:
        df = pd.read_csv(station_file, encoding='utf-8-sig')
        if df.empty: return None, None

        G = nx.Graph()
        
        # 1. 提取站点基本信息（位置与名称）
        nodes_info = df.groupby('station ID').agg({
            '站名': 'first', '经度': 'mean', '纬度': 'mean'
        }).to_dict('index')

        for s_id, info in nodes_info.items():
            G.add_node(s_id, pos=(info['经度'], info['纬度']), label=info['站名'])

        # 2. 生成 P-Space 连边
        # 按“线路”分组，对每一组内的站点 ID 进行两两组合（Combinations）
        for line, group in df.groupby('线路'):
            line_stations = group['station ID'].unique()
            # combinations(n, 2) 会生成该线路下所有站点两两配对的集合
            edges = list(combinations(line_stations, 2))
            G.add_edges_from(edges)
        
        return G, df

    except Exception as e:
        print(f"读取数据或构建模型时出错 {city_name}: {e}")
        return None, None

# -----------------------------------------------------------
# 2. 深度分析与可视化函数
# -----------------------------------------------------------

def visualize_city_pspace(city_name):
    """
    执行 P-Space 拓扑指标计算并生成高度概括的可视化图表。
    注意：由于 P-Space 连边数量远多于 L-Space，计算量会显著增加。
    """
    G, df = load_pspace_data(city_name)
    
    if G is None or G.number_of_nodes() == 0:
        print(f"{city_name} 数据加载失败。")
        return

    # --- A. 拓扑指标计算与专业名词解释 ---
    print("1. 平均聚类系数 (Average Clustering Coefficient)")
    print("[物理意义]：衡量网络局部区域内站点间的通达冗余度与线路集群的紧密程度。")
    print("          若一个站点的聚类系数较高，意味着与其共线的站点之间本身也高度共线（即多条线路在此区域重叠或交织）。")
    print("          代表该区域节点对“换乘”的需求越低，局部线网的鲁棒性与覆盖密度越强。")
    print("2. 接近中心性 (Closeness Centrality)")
    print("[物理意义]：表征站点在全局拓扑结构中的中转效率与平均可达性。接近中心性最高点即为网络的拓扑地理核心。")
    print("          该指标量化了站点作为“全网中转枢纽”的效能该值得分最高的站点，")
    print("          得分越高，代表从该站出发到达全网其他随机站点所需的平均换乘次数越少，体现了该站在系统集成度中的核心地位。")
    print(f"\n--- {city_name} 地铁网 P-Space（换乘逻辑）分析报告 ---")
    
    # 1. 连通性检查
    is_connected = nx.is_connected(G)
    print(f"[连通水平] 全网是否可通过地铁互相到达: {is_connected}")

    # 为了保证路径计算不因网络不连通而报错，提取最大连通子图进行指标分析
    if is_connected:
        G_main = G
    else:
        components = list(nx.connected_components(G))
        G_main = G.subgraph(max(components, key=len)).copy()
        print(f"[数据修正] 由于存在孤岛站点，以下路径指标基于最大核心网（节点数: {G_main.number_of_nodes()}）计算。")

    # 2. 平均聚类系数 (Average Clustering Coefficient)
    # [物理意义]：反映网络中“直达小团体”的密集程度。P-Space 的该值通常极高，接近 1。
    avg_clustering = nx.average_clustering(G)
    print(f"[紧密程度] 平均聚类系数: {avg_clustering:.4f} (数值越高，同线路直达覆盖面越广)")

    # 3. 平均最短路径 (Average Shortest Path Length) 与 直径 (Diameter)
    # [转换公式]：换乘次数 = 路径长度 - 1
    try:
        avg_path = nx.average_shortest_path_length(G_main)
        diameter = nx.diameter(G_main)
        print(f"[出行成本] 平均最短路径长度: {avg_path:.2f} (意味着平均需换乘 {avg_path-1:.2f} 次)")
        print(f"[极限成本] 网络直径: {diameter} (意味着在全网最偏远的两站间旅行，最多需换乘 {diameter-1} 次)")
    except Exception as e:
        print(f"计算路径指标时超时: {e}")

    # 4. 接近中心性 (Closeness Centrality)
    # [物理意义]：谁是“全网换乘之王”。该值得分最高的站点，代表去往全网其他站点所需的平均换乘次数最少。
    close_cent = nx.closeness_centrality(G)
    labels = nx.get_node_attributes(G, 'label')
    top_closeness = sorted(close_cent.items(), key=lambda x: x[1], reverse=True)[:5]
    
    print("\n[全网换乘最便捷站点 Top 5] (到其他站点平均换乘步数最少):")
    for i, (node, val) in enumerate(top_closeness):
        print(f"    第 {i+1} 名: {labels[node]} (评分: {val:.4f})")

    # --- B. 绘图逻辑：动态参数调整 ---
    # P-Space 拥有极高密度的边（例如一条 30 站的线会产生 435 条边），容易导致视觉混乱。
    # 我们根据边总数（Density）动态调整线条宽度和透明度。
    plt.figure(figsize=(12, 10))
    ax = plt.gca()
    pos = nx.get_node_attributes(G, 'pos')
    
    num_edges = G.number_of_edges()
    
    # 策略：边越多，线条越细越淡，以展现网络的宏观结构而非细节纠缠
    if num_edges < 2000:
        e_width, e_alpha = 0.4, 0.25
    elif num_edges < 5000:
        e_width, e_alpha = 0.3, 0.2
    elif num_edges < 500:
        e_width, e_alpha = 0.6, 0.45
    else:
        e_width, e_alpha = 0.25, 0.15

    # 绘制背景连边（直达关系）
    nx.draw_networkx_edges(G, pos, edge_color='royalblue', width=e_width, alpha=e_alpha, ax=ax)
    
    # 绘制站点，使用醒目的红色区分
    nx.draw_networkx_nodes(G, pos, node_size=10, node_color='red', alpha=0.8, ax=ax)

    # 图例配置（仅展示包含的线路名称）
    unique_lines = df['线路'].unique()
    colors = cm.get_cmap('tab20', len(unique_lines))
    legend_elements = [plt.Line2D([0], [0], marker='o', color='w', label=line, 
                                  markerfacecolor=colors(i%20), markersize=8) 
                       for i, line in enumerate(unique_lines)]
    
    # 限制图例展示数量，避免列表过长覆盖图表
    if len(legend_elements) > 25:
        legend_elements = legend_elements[:25]
        legend_elements.append(plt.Line2D([0],[0], marker='', color='w', label='...(更多线路)'))

    ax.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.01, 1), 
              title="覆盖线路列表", fontsize=9, frameon=False)

    plt.title(f"{city_name} 地铁网络 P-Space 拓扑图 (直达/换乘空间分析)", fontsize=18, pad=20)
    plt.axis('equal')
    plt.grid(True, linestyle=':', alpha=0.2)
    plt.tight_layout()
    plt.show()

    print(" P-Space（空间 P）模型与之前使用的 L-Space 模型有本质不同：")
    print(" 1. 定义：在 P-Space 中，只要两个车站属于同一条地铁线路，它们之间就会建立一条直接的“边”。")
    print(" 2. 物理意义：边不再代表一段物理轨道，而是代表“直达可能性”。")
    print(" 3. 路径意义：在 P-Space 中，路径长度为 1 代表直达，为 2 代表需换乘 1 次，以此类推。")
    print(" 4. 选择理由：这种建模方法能够直接反映地铁网络的“换乘便捷性”，是评价乘客心理距离的重要工具。")

print("P-Space 分析引擎已加载完成。")

# ================= Cell 2: 交互式查看 P-Space =================

city_options = get_city_list(DATA_PATH)

if not city_options:
    print("错误：未识别到有效城市数据。")
else:
    print(f"就绪：当前支持对 {len(city_options)} 个城市进行换乘空间（P-Space）评估。")
    
    dropdown = widgets.Dropdown(
        options=city_options,
        value=city_options[0],
        description='选择城市:',
        style={'description_width': 'initial'}
    )

    # 绑定互动组件，切换城市时自动重新计算并绘图
    widgets.interact(visualize_city_pspace, city_name=dropdown);

P-Space 分析引擎已加载完成。
就绪：当前支持对 48 个城市进行换乘空间（P-Space）评估。


interactive(children=(Dropdown(description='选择城市:', options=('Beijing', 'Changchun', 'Changsha', 'Changzhou', …

In [24]:
# ================= Cell 1: P-Space 核心设置与鲁棒性分析函数 =================

# --- 建模逻辑与学术背景 ---
# 1. P-Space 建模：在该模型中，同一条线路上的所有站点被处理为一个“全连通子图”（Clique）。
#    - 物理意义：边代表“直达性”。路径长度为 1 表示无需换乘。
#    - 选择理由：它是评估地铁网络“换乘效率”和“逻辑鲁棒性”的最佳数学模型。
# 2. 鲁棒性（Robustness）分析：通过模拟特定攻击（Targeted Attack）来评估网络崩溃的风险。
#    - 策略：移除网络中“度中心性”最高的 10 个节点（即线路交汇最密集的枢纽）。
#    - 评价指标：全局效率（Global Efficiency）。若移除枢纽后效率大幅下降，说明网络对核心节点的依赖性极强。

%matplotlib inline
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import os
import glob
from itertools import combinations
from matplotlib import cm
import numpy as np
import warnings
import ipywidgets as widgets
from IPython.display import display

# 屏蔽不必要的兼容性警告，确保输出环境的专业性
warnings.filterwarnings('ignore')

# -----------------------------------------------------------
# 1. 核心计算函数
# -----------------------------------------------------------

def get_efficiency(G):
    """
    【名词解释】：网络全局效率 (Global Efficiency)
    【定义】：网络中所有节点对之间最短路径长度倒数的平均值。
    【实际意义】：反映了信息或乘客在网内流动的难易程度。数值越接近 1，代表网内直达性越好，换乘成本越低。
    """
    if len(G) < 2: return 0
    try:
        # 计算公式：E = 1 / [n(n-1)] * sum(1 / d_ij)
        return nx.global_efficiency(G)
    except:
        return 0

# -----------------------------------------------------------
# 2. 深度分析与可视化函数
# -----------------------------------------------------------

def visualize_city_pspace_strict(city_name):
    """
    【功能】：构建 P-Space 模型，执行 Top-10 枢纽节点针对性攻击模拟，并生成动态拓扑图。
    """
    station_file = os.path.join(DATA_PATH, f"{city_name}_All_Stations.csv")
    
    if not os.path.exists(station_file):
        print(f"[警告] 数据文件缺失，无法分析: {city_name}")
        return

    print("网络全局效率 (Global Efficiency)")
    print("[专业定义]：基于 P-Space 拓扑测度，定义为网络中所有节点对之间最短换乘路径距离倒数的算术平均值。")
    print("[物理意义]：量化轨道交通线网在拓扑层面的全局换乘通达性与系统集成效率。")
    print("         在 P-Space 模型中，相邻节点代表处于同一运行线路，无需换乘。")
    print("         因此，全局效率反映了乘客在网内随机两站间旅行时，规避中转开销的整体能力。")
    print("         数值越接近 1，表明线网的直接互联性越强，通过零换乘或极低换乘实现全网覆盖的性能越优，体现了系统高度的拓扑紧凑性。")

    try:
        df = pd.read_csv(station_file, encoding='utf-8-sig')
        if df.empty: return

        # --- 步骤 A：构建 P-Space 拓扑图 ---
        G_P = nx.Graph()
        nodes_info = df.groupby('station ID').agg({
            '站名': 'first', '经度': 'mean', '纬度': 'mean'
        }).to_dict('index')

        for s_id, info in nodes_info.items():
            G_P.add_node(s_id, pos=(info['经度'], info['纬度']), label=info['站名'])

        # P-Space 连边逻辑：同线即连通
        for line, group in df.groupby('线路'):
            line_stations = group['station ID'].unique()
            G_P.add_edges_from(combinations(line_stations, 2))

        # --- 步骤 B：辅助识别物理枢纽 (用于可视化分级) ---
        # 即使在 P-Space 分析中，我们也引用 L-Space 的物理连接数来定义“物理枢纽”
        G_temp = nx.Graph()
        for line, group in df.groupby('线路'):
            line_stations = group['station ID'].unique()
            for i in range(len(line_stations)-1):
                G_temp.add_edge(line_stations[i], line_stations[i+1])
        
        # 物理换乘站：在实际地理连接中度数 > 2 的车站
        transfer_stations = [n for n, d in G_temp.degree() if d > 2 and n in G_P.nodes()]
        normal_stations = [n for n in G_P.nodes() if n not in transfer_stations]

        # --- 步骤 C：鲁棒性压力测试报告 ---
        print(f"\n--- {city_name} 地铁网络 P-Space 鲁棒性深度评估 ---")
        
        # 计算 P-Space 逻辑下的度分布
        degrees_p = [d for n, d in G_P.degree()]
        if degrees_p:
            print(f"[网络特征] 最大逻辑连接数: {max(degrees_p)} | 最小逻辑连接数: {min(degrees_p)}")
        
        # 1. 初始状态效率
        print("[分析中] 正在计算初始状态下的网络全局效率...")
        initial_eff = get_efficiency(G_P)
        
        # 2. 模拟针对性攻击：移除度数（逻辑连接数）最高的 10 个枢纽站
        top_hubs = sorted(dict(G_P.degree()).items(), key=lambda x: x[1], reverse=True)[:10]
        G_attacked = G_P.copy()
        G_attacked.remove_nodes_from([n[0] for n in top_hubs])
        
        # 3. 故障状态效率计算
        print("[分析中] 正在计算核心节点失效后的网络效率...")
        attacked_eff = get_efficiency(G_attacked)
        
        # 4. 损失系数计算
        # 该比例反映了这 10 个车站对整个网络换乘便捷性的贡献占比
        drop_ratio = ((initial_eff - attacked_eff) / initial_eff) * 100 if initial_eff > 0 else 0

        print(f"[核心指标] 初始网络效率: {initial_eff:.4f}")
        print(f"[核心指标] 针对性攻击后效率: {attacked_eff:.4f}")
        print(f"[鲁棒性评估] 网络效率下降比例: {drop_ratio:.2f}% (该值越高，说明网络对核心节点的依赖度越高)")
        
        labels = nx.get_node_attributes(G_P, 'label')
        print("[关键目标] 移除的 Top-5 核心逻辑枢纽站:")
        for node, val in top_hubs[:5]:
             print(f"    - {labels.get(node, node)} (逻辑连通数: {val})")

        # --- 步骤 D：动态可视化展示 ---
        plt.figure(figsize=(12, 10))
        ax = plt.gca()
        pos = nx.get_node_attributes(G_P, 'pos')

        # 【核心设计】：根据连边总数自动调节绘图精细度，防止特大城市图表过度拥挤
        num_edges = G_P.number_of_edges()
        if num_edges < 500:          # 极其稀疏
            e_width, e_alpha = 0.6, 0.45
        elif num_edges < 2000:       # 中等规模
            e_width, e_alpha = 0.4, 0.25
        elif num_edges < 5000:       # 较大规模
            e_width = 0.3
            e_alpha = 0.2
        else:                        # 极大规模 (如北京、上海)
            e_width, e_alpha = 0.25, 0.15

        # 绘制换乘逻辑连边
        nx.draw_networkx_edges(G_P, pos, edge_color='royalblue', width=e_width, alpha=e_alpha, ax=ax)

        # 绘制站点，利用节点大小区分物理层级
        nx.draw_networkx_nodes(G_P, pos, nodelist=normal_stations, 
                             node_size=10, node_color='red', alpha=0.4, ax=ax)
        nx.draw_networkx_nodes(G_P, pos, nodelist=transfer_stations, 
                             node_size=20, node_color='red', alpha=0.9, ax=ax)

        plt.title(f"{city_name} P-Space 换乘逻辑拓扑图 (直达连通性可视化)", fontsize=18, pad=20)
        
        # 线路图例生成
        unique_lines = df['线路'].unique()
        colors = cm.get_cmap('tab20', len(unique_lines))
        max_legends = 30
        legend_lines = unique_lines[:max_legends]
        legend_elements = [plt.Line2D([0], [0], marker='o', color='w', label=line, 
                                      markerfacecolor=colors(i%20), markersize=8) 
                           for i, line in enumerate(legend_lines)]
        
        if len(unique_lines) > max_legends:
            legend_elements.append(plt.Line2D([0],[0], marker='', color='w', label='... 更多线路'))

        ax.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.02, 1), 
                  title="涉及地铁线路", fontsize=10, frameon=False)

        plt.axis('equal')
        plt.grid(True, linestyle=':', alpha=0.2)
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"[错误] 处理城市 {city_name} 时发生逻辑异常: {e}")

    print(" 1. P-Space 建模：在该模型中，同一条线路上的所有站点被处理为一个“全连通子图”（Clique）。")
    print("- 物理意义：边代表“直达性”。路径长度为 1 表示无需换乘。")
    print("- 选择理由：它是评估地铁网络“换乘效率”和“逻辑鲁棒性”的最佳数学模型。")
    print("2. 鲁棒性（Robustness）分析：通过模拟特定攻击（Targeted Attack）来评估网络崩溃的风险。")
    print("- 策略：移除网络中“度中心性”最高的 10 个节点（即线路交汇最密集的枢纽）。")
    print("- 评价指标：全局效率（Global Efficiency）。若移除枢纽后效率大幅下降，说明网络对核心节点的依赖性极强。")

print("P-Space 深度分析引擎已就绪。")

# ================= Cell 2: 交互式查看系统 =================

city_options = get_city_list(DATA_PATH)

if not city_options:
    print("[错误] 未能识别到任何城市数据文件。")
else:
    print(f"[信息] 当前分析环境已成功加载 {len(city_options)} 个城市的地铁模型。")
    
    dropdown = widgets.Dropdown(
        options=city_options,
        value=city_options[0],
        description='选择分析城市:',
        style={'description_width': 'initial'}
    )

    # 运行交互界面
    widgets.interact(visualize_city_pspace_strict, city_name=dropdown);

P-Space 深度分析引擎已就绪。
[信息] 当前分析环境已成功加载 48 个城市的地铁模型。


interactive(children=(Dropdown(description='选择分析城市:', options=('Beijing', 'Changchun', 'Changsha', 'Changzhou'…

In [28]:
# ================= Cell: 多城市交互式 BCDE 深度分析 (表格+图表) =================

# --- 逻辑说明 ---
# 本代码块实现了对地铁网络“核心节点”的全方位画像。
# 我们引入了复杂网络中最重要的四个中心性指标（简称 BCDE），用于从不同维度定义“谁是核心站”：
# 1. Degree (度中心性)：衡量站点的直接连接数，反映最基础的连通性。
# 2. Betweenness (中介中心性)：衡量站点在所有最短路径中出现的频率，反映其作为“咽喉枢纽”的控制力。
# 3. Closeness (接近中心性)：衡量站点到全网其他所有站点的平均距离，反映其作为“出行中心”的便捷性。
# 4. Eigenvector (特征向量中心性)：衡量站点所连接的邻居站点的质量，反映其“朋友圈”的权重。

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML

def run_full_city_analysis(city_name):
    """
    【功能】：对指定城市进行全套复杂网络深度分析。
    【输出】：包含全局效能报告、四大维度排行榜、以及多维度空间分布可视化图组。
    """
    # 1. 加载数据并预处理
    G, _ = load_data(city_name)
    if G is None or G.number_of_nodes() == 0:
        print(f"无法获取城市 {city_name} 的有效数据。")
        return

    # 为了确保直径（Diameter）和路径长度计算的数学严谨性，必须提取最大连通子图进行计算
    # 理由：如果网络存在孤岛站点，全局路径长度在数学上会趋于无穷大
    largest_cc = max(nx.connected_components(G), key=len)
    G_main = G.subgraph(largest_cc).copy()
    
    labels = nx.get_node_attributes(G_main, 'label')
    pos = nx.get_node_attributes(G_main, 'pos')
    
    # -------------------------------------------------------
    # 2. 计算模块：网络全局指标与节点中心性
    # -------------------------------------------------------
    print(f"正在深度分析城市: {city_name}  ...")
    
    # 全局基础指标计算
    try:
        # 直径：衡量城市地铁通行的“最长极限成本”
        diameter = nx.diameter(G_main)
        # 聚类系数：反映区域轨道网的小团体化程度，数值越高代表局部通达性越好
        avg_clustering = nx.average_clustering(G_main)
        # 平均最短路径：乘客在任意两站间旅行平均需要经过的站数
        avg_path = nx.average_shortest_path_length(G_main)
        # 密度：描述地铁网发育的成熟度，数值越高代表网格越密集而非放射状
        density = nx.density(G_main)
    except:
        diameter, avg_path, avg_clustering, density = "计算受阻", "计算受阻", "计算受阻", "计算受阻"

    # 计算 BCDE 中心性
    deg_cent = nx.degree_centrality(G_main)      # D：直接连通性
    bet_cent = nx.betweenness_centrality(G_main) # B：中介控制力
    clo_cent = nx.closeness_centrality(G_main)   # C：便捷通达度
    try:
        # 特征向量中心性：反映站点的影响力（连接了其他重要站点的站，得分更高）
        eig_cent = nx.eigenvector_centrality(G_main, max_iter=2000) # E：地位权重
    except:
        eig_cent = {n: 0 for n in G_main.nodes()}

    # -------------------------------------------------------
    # 3. 输出展示：全局指标报告 (HTML 格式)
    # -------------------------------------------------------
    display(HTML(f"<h2 style='color: #2e76b1; border-bottom: 2px solid #2e76b1;'>{city_name} 地铁网全局效能分析报告</h2>"))
    
    metrics_html = f"""
    <div style="background-color: #f7f7f7; padding: 15px; border-radius: 8px; border: 1px solid #cccccc; line-height: 1.8;">
        <strong>[网络规模]</strong> 车站总数: {len(G_main)} 个 | 轨道段数: {G_main.number_of_edges()} | 网络密集度: {density:.6f}<br>
        <strong>[传输效能]</strong> 网络直径: {diameter} 站 (穿越城市最远两端的成本) | 平均出行站数: {avg_path:.4f} 站<br>
        <strong>[连通特征]</strong> 平均聚类系数: {avg_clustering:.4f} (反映局部区域的直达紧密度)
    </div>
    """
    display(HTML(metrics_html))

    # -------------------------------------------------------
    # 4. 输出展示：分维度排行榜 (数据表形式)
    # -------------------------------------------------------
    data_rows = []
    for node in G_main.nodes():
        data_rows.append({
            '站点名称': labels.get(node, str(node)),
            '度中心性': deg_cent[node],
            '中介中心性': bet_cent[node],
            '接近中心性': clo_cent[node],
            '特征向量中心性': eig_cent[node]
        })
    full_df = pd.DataFrame(data_rows)

    display(HTML("<h3 style='margin-top: 25px;'>网络关键节点排行榜 (Top 10)</h3>"))
    
    # 针对四个维度分别输出 Top 10，并附带专业解释
    metrics_list = [
        ('中介中心性', '衡量站点作为咽喉枢纽的控制力，失效后对全网影响最大'),
        ('接近中心性', '衡量站点去往全网各处的平均便捷程度'),
        ('度中心性', '衡量站点的直接连接线路数，是基础换乘能力的体现'),
        ('特征向量中心性', '衡量站点在网络中的整体地位及影响力评分')
    ]

    for col_name, description in metrics_list:
        top_df = full_df[['站点名称', col_name]].sort_values(by=col_name, ascending=False).head(10).reset_index(drop=True)
        top_df.index = top_df.index + 1
        
        display(HTML(f"<p style='background: #e1eaf2; padding: 5px;'><strong>维度: {col_name}</strong> - {description}</p>"))
        display(top_df)
        print("-" * 80)

    # -------------------------------------------------------
    # 5. 可视化绘制：2x2 空间分布对照图
    # -------------------------------------------------------
    # 映射标题：将技术名词转换为物理意义标题
    metrics_map = {
        '中介分布 (Hubs 控制力)': bet_cent,
        '接近分布 (Access 便捷度)': clo_cent,
        '度分布 (Degree 连接数)': deg_cent,
        '地位权重 (Prestige 影响力)': eig_cent
    }

    fig, axes = plt.subplots(2, 2, figsize=(18, 16))
    axes = axes.flatten()
    
    # 标注偏移量预设，用于防止站名标签重叠
    label_offsets = [(15, 15), (-30, 20), (25, -25), (-30, -25), (0, 35)]

    for i, (title, metric_dict) in enumerate(metrics_map.items()):
        ax = axes[i]
        nodelist = list(G_main.nodes())
        values = [metric_dict[n] for n in nodelist]
        
        # 归一化处理：使节点大小映射到 40 到 400 之间，突出显示核心节点
        v_min, v_max = min(values), max(values)
        node_sizes = [(v - v_min)/(v_max - v_min + 1e-9) * 360 + 40 for v in values]

        # 绘制背景边
        nx.draw_networkx_edges(G_main, pos, alpha=0.1, edge_color='#888888', ax=ax)
        # 绘制节点：颜色深浅代表数值高低，大小代表权重
        sc = nx.draw_networkx_nodes(G_main, pos, nodelist=nodelist,
                                    node_size=node_sizes, node_color=values,
                                    cmap=plt.cm.viridis, alpha=0.8, ax=ax)
        
        # 配置颜色映射条
        cbar = plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.04)
        cbar.ax.set_ylabel('重要度评分', fontsize=10)
        
        # 精准标注 Top 5 站点：通过引线方式在地图上指出最关键的枢纽
        top_5_nodes = sorted(metric_dict.items(), key=lambda x: x[1], reverse=True)[:5]
        for idx, (node_id, val) in enumerate(top_5_nodes):
            x, y = pos[node_id]
            st_name = labels.get(node_id, str(node_id))
            
            ax.annotate(
                st_name,
                xy=(x, y), 
                xytext=label_offsets[idx % len(label_offsets)], 
                textcoords='offset points',
                fontsize=11, fontweight='bold',
                bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', boxstyle='round,pad=0.3'),
                arrowprops=dict(arrowstyle='->', connectionstyle="arc3,rad=0.1", color='#cc0000', alpha=0.6)
            )

        ax.set_title(f"{city_name}: {title}", fontsize=16, fontweight='bold', pad=15)
        ax.axis('off') # 隐藏地理坐标轴，突出拓扑结构

    plt.suptitle(f"{city_name} 地铁网 BCDE 多维度中心性拓扑分布图集", fontsize=24, y=0.98)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

    print(" 本代码块实现了对地铁网络“核心节点”的全方位画像。")
    print("我们引入了复杂网络中最重要的四个中心性指标（简称 BCDE），用于从不同维度定义“谁是核心站”：")
    print("1. Degree (度中心性)：衡量站点的直接连接数，反映最基础的连通性。")
    print("2. Betweenness (中介中心性)：衡量站点在所有最短路径中出现的频率，反映其作为“咽喉枢纽”的控制力。")
    print("3. Closeness (接近中心性)：衡量站点到全网其他所有站点的平均距离，反映其作为“出行中心”的便捷性。")
    print("4. Eigenvector (特征向量中心性)：衡量站点所连接的邻居站点的质量，反映其“朋友圈”的权重。")

# -----------------------------------------------------------
# 交互式组件部署：启动城市选择器
# -----------------------------------------------------------
city_options = get_city_list(DATA_PATH)

if not city_options:
    print("未发现城市数据文件，请检查 DATA_PATH 路径配置是否正确。")
else:
    # 定义选择菜单
    dropdown = widgets.Dropdown(
        options=city_options,
        value=city_options[0],
        description='分析对象:',
        style={'description_width': 'initial'}
    )

    # 建立互动绑定：切换城市时，全套分析报告将自动重新生成
    widgets.interact(run_full_city_analysis, city_name=dropdown);

interactive(children=(Dropdown(description='分析对象:', options=('Beijing', 'Changchun', 'Changsha', 'Changzhou', …

In [29]:
# ================= Cell: 线路级失效综合对比分析 (双指标柱状图) =================

# --- 模拟逻辑说明 ---
# 本分析模块用于评估地铁网络对线路失效的敏感程度。
# 1. 模拟过程：依次从网络中移除某条特定线路的所有区间（边），但保留站点（节点）。
# 2. 观测指标：
#    - 全局效率 (Global Efficiency)：反映网络中“出行效能”的损失，即移除线路后，剩下的站与站之间平均变远了多少。
#    - 最大连通分量比例 (Largest Connected Component Ratio)：反映网络物理上“分崩离析”的程度。
# 3. 选择理由：这种对比分析能帮助我们识别哪些线路是“效率支撑者”（影响速度），哪些是“结构连接者”（防止网络分裂）。

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML

def simulate_line_failures_combined(city_name):
    """
    [功能]：模拟地铁线路级失效，并将出行效率与物理连通性两个维度的损失整合对比。
    [物理意义]：识别该城市对整体路网贡献最大的“关键线路”。
    """
    # 1. 环境准备与基准建立
    G_raw, edges_df = load_data(city_name)
    if G_raw is None or edges_df is None: 
        print(f"[错误] 无法加载城市 {city_name} 的数据。")
        return

    # 为确保指标的数学一致性，首先提取初始状态下的最大核心网（最大连通子图）
    lcc_nodes = max(nx.connected_components(G_raw), key=len)
    G_base = G_raw.subgraph(lcc_nodes).copy()
    N_total = G_base.number_of_nodes()
    
    # 计算初始状态的基准值
    base_s = 1.0 # 初始核心网连通比例定义为 100%
    base_e = nx.global_efficiency(G_base) # 初始出行效能基准
    
    unique_lines = edges_df['线路'].unique()
    line_results = []

    # -------------------------------------------------------
    # 2. 循环移除模拟：遍历每一条线路
    # -------------------------------------------------------
    for target_line in unique_lines:
        # 核心逻辑：从数据集中剔除属于该线路的轨道边
        remaining_edges = edges_df[edges_df['线路'] != target_line]
        
        # 重新构建移除边之后的网络拓扑
        G_temp = nx.Graph()
        G_temp.add_nodes_from(G_base.nodes())
        for _, row in remaining_edges.iterrows():
            if row['from_id'] in G_temp and row['to_id'] in G_temp:
                G_temp.add_edge(row['from_id'], row['to_id'])
        
        # 如果移除后依然有轨道存在，计算失效后的性能表现
        if G_temp.number_of_edges() > 0:
            # 计算当前最大的核心网包含多少比例的车站 (S指标)
            current_lcc = len(max(nx.connected_components(G_temp), key=len)) / N_total
            # 计算当前通行的全局效率 (E指标)
            current_eff = nx.global_efficiency(G_temp)
        else:
            current_lcc, current_eff = 0, 0
            
        # 计算损失百分比：数值越大，说明该线路越重要
        s_drop = (base_s - current_lcc) * 100
        e_drop = (base_e - current_eff) / base_e * 100
        
        line_results.append({
            "线路名称": target_line,
            "效率损失 (E-Drop %)": e_drop,
            "连通性损失 (S-Drop %)": s_drop
        })

    # 3. 结果排序：按“效率损失”从高到低排序，识别最核心的线路
    df_res = pd.DataFrame(line_results).sort_values(by="效率损失 (E-Drop %)", ascending=False)

    # -------------------------------------------------------
    # 4. 可视化绘制：组合条形图 (学术严谨风格)
    # -------------------------------------------------------
    plt.figure(figsize=(14, 7))
    x = np.arange(len(df_res)) 
    width = 0.35 # 条形柱宽度

    # 绘制双指标对比图
    # 红色代表效率损失：反映乘客绕路成本
    rects1 = plt.bar(x - width/2, df_res['效率损失 (E-Drop %)'], width, 
                     label='出行效率损失 (E)', color='#d62728', alpha=0.8) 
    # 蓝色代表连通性损失：反映网络萎缩程度
    rects2 = plt.bar(x + width/2, df_res['连通性损失 (S-Drop %)'], width, 
                     label='网络连通损失 (S)', color='#1f77b4', alpha=0.6) 

    # 标注与装饰
    plt.title(f"{city_name} 地铁线路级失效风险分析报告 (脆弱性评估)", fontsize=16, pad=20)
    plt.ylabel("性能下降比例 (%)", fontsize=12)
    plt.xlabel("线路名称", fontsize=12)
    plt.xticks(x, df_res['线路名称'], rotation=45, ha='right')
    plt.legend(frameon=False, fontsize=11)
    plt.grid(axis='y', linestyle=':', alpha=0.5)

    # 针对规模适中的图表添加数据标签，提高易读性
    if len(df_res) < 20:
        for rect in rects1:
            height = rect.get_height()
            plt.annotate(f'{height:.1f}%', xy=(rect.get_x() + rect.get_width() / 2, height),
                         xytext=(0, 3), textcoords="offset points", ha='center', fontsize=9)

    plt.tight_layout()
    plt.show()

    # -------------------------------------------------------
    # 5. 指标名词解释与排行输出
    # -------------------------------------------------------
    explanation_html = """
    <div style="padding: 10px; border: 1px solid #ddd; border-left: 5px solid #2e76b1; background-color: #f9f9f9;">
        <strong>指标名词解释：</strong><br>
        1. <strong>效率损失 (E-Drop)</strong>：衡量该线路失效后，全网平均换乘步数增加的情况。数值越高，该线对缩短全网通勤时间贡献越大。<br>
        2. <strong>连通性损失 (S-Drop)</strong>：衡量该线路失效后，网络是否分裂成互不相通的孤岛。数值越高，该线在维持网络拓扑完整性上越关键。
    </div>
    """
    display(HTML(f"<h3>{city_name} 线路失效影响排行榜 (前 10 名)</h3>"))
    display(HTML(explanation_html))
    display(df_res.head(10).reset_index(drop=True))

    print("本分析模块用于评估地铁网络对线路失效的敏感程度。")
    print("1. 模拟过程：依次从网络中移除某条特定线路的所有区间（边），但保留站点（节点）。")
    print("2. 观测指标：")
    print("- 全局效率 (Global Efficiency)：反映网络中“出行效能”的损失，即移除线路后，剩下的站与站之间平均变远了多少。")
    print("- 最大连通分量比例 (Largest Connected Component Ratio)：反映网络物理上“分崩离析”的程度。")
    print("3. 选择理由：这种对比分析能帮助我们识别哪些线路是“效率支撑者”（影响速度），哪些是“结构连接者”（防止网络分裂）。")

# --- 部署交互式组件 ---
city_options = get_city_list(DATA_PATH)

if not city_options:
    print("[错误] 未发现数据路径，请检查 DATA_PATH 配置。")
else:
    dropdown_line_comb = widgets.Dropdown(
        options=city_options, 
        value=city_options[0],
        description='城市选择:',
        style={'description_width': 'initial'}
    )
    widgets.interact(simulate_line_failures_combined, city_name=dropdown_line_comb);

interactive(children=(Dropdown(description='城市选择:', options=('Beijing', 'Changchun', 'Changsha', 'Changzhou', …

In [44]:
import random
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import ipywidgets as widgets

# --- 全局专业名词报告 (保持不变) ---
def print_robustness_basics():
    print("-" * 30 + " 鲁棒性模拟基础名词解释 " + "-" * 30)
    print("[1. 蓄意攻击 (Targeted Attack)]")
    print("[专业表述]：基于节点中心性序位的确定性移除策略 (Deterministic Node Removal)")
    print("[物理意义]：模拟具备全局拓扑知识的“智能化”破坏行为。通过优先剥离网络中具有最高权重值的关键节点，揭示轨道交通系统在精确打击下的拓扑脆弱性 (Vulnerability)。")
    print("          这是识别系统“单点故障”影响力的关键手段。即根据节点在网络中的重要性（如换乘站权重）进行有目的的移除")
    print("\n[2. 随机失效 (Random Failure)]")
    print("[专业表述]：基于均匀概率分布的随机节点渗流过程 (Random Site Percolation)。")
    print("[物理意义]：模拟系统中离散、无关联的随机扰动（如设备老化、偶发性供电故障）。即模拟无差别地随机关闭车站，模拟设备老化、突发事故等日常风险。")
    print("          在稳健性分析中，随机失效作为评估基准 (Baseline)，用于量化针对性攻击对系统结构崩溃的加速效应 (Acceleration Effect)。")
    print("          即衡量针对性攻击对系统造成的额外破坏力。")
    print("\n[3. 渗流理论 (Percolation Theory)]")
    print("[专业表述]：研究网络在组分移除过程中的相变 (Phase Transition) 与巨分量演化规律。")
    print("   核心逻辑：研究当节点不断失效时，网络连通性何时发生突变（即‘崩塌’）。")
    print("[物理意义]：在城市轨道交通韧性研究中，它定义了系统在遭遇连锁失效前所能耐受的受损阈值上限。")
    print("          该理论用于探测网络从“全局连通状态”向“离散碎片状态”转化的临界动力学。")
    print("          在城市轨道交通韧性研究中，它定义了系统在遭遇连锁失效前所能耐受的受损阈值上限。即帮助确定城市交通网在彻底瘫痪前能承受的最大受损比例。")
    print("\n[4.最大连通分量比例 (Proportion of the Largest Connected Component, S)]")
    print("[专业表述]：量化系统宏观拓扑完整性的序参数 (Order Parameter)。")
    print("[物理意义]：S 值定义为当前网络中最大连通子图（巨分量）包含的节点数与原始网络节点总数的比值。它表征了系统在遭受扰动后，")
    print("          仍能维持跨区域通达能力的骨干结构规模。当 S 发生阶跃式下降时，标志着网络已进入彻底碎片化的结构相变点。")
    print("-" * 84 + "\n")

def simulate_robustness(city_name):
    """
    【功能】：模拟地铁网络在遭受蓄意攻击与随机失效下的崩溃过程。
    【修复】：对齐 X 轴物理意义，修正百分比统计逻辑，引入随机均值基准。
    """
    # 1. 基础数据准备
    G_orig, _ = load_data(city_name)
    if G_orig is None: return
    
    # 确保实验起点为 100% 物理连通
    lcc_nodes = max(nx.connected_components(G_orig), key=len)
    G = G_orig.subgraph(lcc_nodes).copy()
    N = G.number_of_nodes()
    node_labels = nx.get_node_attributes(G, 'label')
    
    # -----------------------------------------------------------
    # 2. 内部函数：网络崩塌模拟器 (修改：支持全过程模拟)
    # -----------------------------------------------------------
    def get_collapse_data(node_list):
        temp_G = G.copy()
        s_curve = [1.0] 
        removed_nodes_info = []
        threshold_stats = {}
        
        # 修改：循环至 N，确保覆盖 0%-100% 全过程
        for i in range(1, N + 1):
            node_to_remove = node_list[i-1]
            if temp_G.has_node(node_to_remove):
                temp_G.remove_node(node_to_remove)
            
            if i <= 10: 
                removed_nodes_info.append(node_labels.get(node_to_remove, str(node_to_remove)))
            
            if temp_G.number_of_nodes() > 0:
                comps = list(nx.connected_components(temp_G))
                largest_cc_size = len(max(comps, key=len))
                s = largest_cc_size / N
            else:
                s = 0
            s_curve.append(s)
            
            # 记录关键百分比下的生存状况 (此时能准确匹配到 75% 等位置)
            for pct in [5, 10, 25, 50, 75]:
                if i == int(N * pct / 100): 
                    threshold_stats[f"{pct}%"] = s
        return s_curve, removed_nodes_info, threshold_stats

    # --- 步骤 1: 建立稳定的随机失效基准 (修改：多次模拟取均值) ---
    print(f"--- {city_name} 鲁棒性压力测试运行中 ---")
    random_runs = 10
    all_random_curves = []
    for _ in range(random_runs):
        nodes_rand = list(G.nodes())
        random.shuffle(nodes_rand)
        curve, _, _ = get_collapse_data(nodes_rand)
        all_random_curves.append(curve)
    random_curve_mean = np.mean(all_random_curves, axis=0)

    # --- 步骤 2: 四大中心性攻击对比 ---
    for attack_metric in ['Betweenness', 'Degree', 'Closeness', 'Eigenvector']:
        
        if attack_metric == 'Betweenness': 
            cent = nx.betweenness_centrality(G)
        elif attack_metric == 'Degree': 
            cent = nx.degree_centrality(G)
        elif attack_metric == 'Closeness': 
            cent = nx.closeness_centrality(G)
        elif attack_metric == 'Eigenvector':
            try: cent = nx.eigenvector_centrality(G, max_iter=2000)
            except: cent = nx.degree_centrality(G)
        
        nodes_targeted = sorted(cent, key=cent.get, reverse=True)
        targeted_curve, top_nodes, targeted_stats = get_collapse_data(nodes_targeted)

        # --- 步骤 3: 打印分析结果报告 (修改：对齐 75% 数据) ---
        print(f"\n[{attack_metric} 攻击报告]")
        print(f"1. 核心致损点 (Top 10): {' -> '.join(top_nodes)}")
        print(f"2. 渗流阈值分析 S值：")
        print(f"   - 移除 5% 节点后: {targeted_stats.get('5%', 0):.4f}")
        print(f"   - 移除 10% 节点后: {targeted_stats.get('10%', 0):.4f}")
        print(f"   - 移除 25% 节点后: {targeted_stats.get('25%', 0):.4f}")
        print(f"   - 移除 50% 节点后: {targeted_stats.get('50%', 0):.4f}")
        print(f"   - 移除 75% 节点后: {targeted_stats.get('75%', 0):.4f}")

        # --- 步骤 4: 可视化绘制 (修改：修复 X 轴映射错误) ---
        plt.figure(figsize=(10, 5))
        
        # 修复：x_axis 的每一个点对应实际移除的百分比
        x_axis = np.array(range(len(targeted_curve))) / N * 100
        
        plt.plot(x_axis, random_curve_mean, label=f'Random Failure ({random_runs} runs mean)', 
                 color='green', lw=2, linestyle='--', alpha=0.6)
        plt.plot(x_axis, targeted_curve, label=f'Targeted Attack ({attack_metric})', 
                 color='red', lw=2.5)
        
        plt.title(f"Percolation Analysis: {city_name} ({attack_metric})", fontsize=14, fontweight='bold')
        plt.xlabel("Actual Nodes Removed (%)", fontsize=12)
        # 将原来的 Relative Size 改为 Proportion
        plt.ylabel("LCC Proportion ($S$)", fontsize=12)
        plt.xlim(0, 100) # 固定显示 0-100%
        plt.ylim(0, 1.05)
        plt.grid(True, linestyle=':', alpha=0.5)
        plt.legend(frameon=False)
        plt.tight_layout()
        plt.show()
        print("-" * 92)

    print("\n[方案评估]")
    print("优点 (Advantages):")
    print("1. 指标针对性强：通过对比四类拓扑属性，能精准定位对网络生存贡献最大的站点类型。")
    print("2. 基准参考价值：随机失效曲线为评估人为破坏的严重程度提供了客观参照。")
    print("局限性 (Limitations):")
    print("1. 静态评估局限：攻击序列在模拟开始前已确定，未考虑网络受损后动态调整的复杂性。")
    print("2. 仅关注物理连通：当前的 S值 仅反映了结构拓扑，未纳入客流负载等流量指标。")
    
    print("\n[图形走势学术解释]")
    print("解释：红色实线（蓄意攻击）相较于绿色虚线（随机失效）下降越快，说明该网络对核心枢纽站的依赖性越强。")
    print("如果曲线出现‘断崖式’下跌，标志着网络发生了渗流转变，系统从整体运营转为彻底碎片化瘫痪。")

# --- 部署交互界面 ---
city_list = get_city_list(DATA_PATH)
if city_list:
    print_robustness_basics()
    widgets.interact(
        simulate_robustness, 
        city_name=widgets.Dropdown(options=city_list, description='分析城市:')
    )

------------------------------ 鲁棒性模拟基础名词解释 ------------------------------
[1. 蓄意攻击 (Targeted Attack)]
[专业表述]：基于节点中心性序位的确定性移除策略 (Deterministic Node Removal)
[物理意义]：模拟具备全局拓扑知识的“智能化”破坏行为。通过优先剥离网络中具有最高权重值的关键节点，揭示轨道交通系统在精确打击下的拓扑脆弱性 (Vulnerability)。
          这是识别系统“单点故障”影响力的关键手段。即根据节点在网络中的重要性（如换乘站权重）进行有目的的移除

[2. 随机失效 (Random Failure)]
[专业表述]：基于均匀概率分布的随机节点渗流过程 (Random Site Percolation)。
[物理意义]：模拟系统中离散、无关联的随机扰动（如设备老化、偶发性供电故障）。即模拟无差别地随机关闭车站，模拟设备老化、突发事故等日常风险。
          在稳健性分析中，随机失效作为评估基准 (Baseline)，用于量化针对性攻击对系统结构崩溃的加速效应 (Acceleration Effect)。
          即衡量针对性攻击对系统造成的额外破坏力。

[3. 渗流理论 (Percolation Theory)]
[专业表述]：研究网络在组分移除过程中的相变 (Phase Transition) 与巨分量演化规律。
   核心逻辑：研究当节点不断失效时，网络连通性何时发生突变（即‘崩塌’）。
[物理意义]：在城市轨道交通韧性研究中，它定义了系统在遭遇连锁失效前所能耐受的受损阈值上限。
          该理论用于探测网络从“全局连通状态”向“离散碎片状态”转化的临界动力学。
          在城市轨道交通韧性研究中，它定义了系统在遭遇连锁失效前所能耐受的受损阈值上限。即帮助确定城市交通网在彻底瘫痪前能承受的最大受损比例。

[4.最大连通分量比例 (Proportion of the Largest Connected Component, S)]
[专业表述]：量化系统宏观拓扑完整性的序参数 (Order Parameter)。
[物理意义]：S 值定义为当前网络中最大连通子图（巨分

interactive(children=(Dropdown(description='分析城市:', options=('Beijing', 'Changchun', 'Changsha', 'Changzhou', …

In [58]:
# ================= 代码段 2: 综合鲁棒性深度评估系统 (全阈值快照对比版) =================

import random
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import ipywidgets as widgets
from matplotlib import cm

# --- 1. 全局专业名词定义报告 (学术审核版 - 保持原样) ---
def print_terminology_report():
    print("-" * 30 + " 专业名词与参数定义说明 " + "-" * 30)
    print("[1. 宏观拓扑完整性 (Size of the Largest Connected Component, S值)]")
    print(" [专业定义]：受损网络中最大连通子图（巨分量LCC）包含的节点数与初始网络总节点数的比值。")
    print(" [物理意义]：作为渗流理论中的序参数 (Order Parameter)，S值量化了地铁系统物理骨架的宏观连通水平。反映地铁系统的‘物理骨骼’是否维持全局连通。")
    print("           它表征了系统在遭受扰动后，仍能维持跨区域通达能力的骨干结构规模。")
    print("           当 S值 发生阶跃式下降、跌破临界点意味着网络发生相变时，标志着网络已进入彻底碎片化的结构相变点。")
    print("\n[2. 全局传输效能 (Global Efficiency, E值)]")
    print("[专业定义]：网络中所有节点对之间最短路径长度倒数的算术平均值。")
    print("[物理意义]：衡量系统功能的有效性。作为系统功能的有效性测度，反映了线网在遭受破坏过程中的性能降级 (Performance Degradation)。")
    print("           由于该指标考虑了路径长度的倒数，它能够有效评估网络在分裂为多个孤岛后，残余结构对于乘客通勤的平均通达效率。")
    print("           即使物理骨架（S值）尚存，若核心枢纽失效导致绕路开销剧增，E值仍会显著下降，揭示了网络服务水平的丧失。")
    print("           因此这反映乘客在网内通勤的通达便捷程度；若关键枢纽失效导致绕路开销剧增，E值将显著下降。")
    print("\n[3. 累积鲁棒性指数 (Robustness Index, R值)]")
    print("[专业定义]：性能衰减曲线（S或E指标）在节点移除全过程中的累积积分面积，即与坐标轴围成的积分面积(Area Under Curve, AUC)。")
    print(" [物理意义]: 量化系统在遭受‘连续性打击’全过程中的平均生存效能。R值将瞬时的崩溃转折与长期的衰减过程整合为一个标量,数值范围 [0, 0.5]。")
    print("           数值越大代表系统韧性越优，即数值越大代表了系统在连续移除压力下表现出的拓扑稳健性与抗打击能力越优。")
    print("\n[4. 蓄意攻击 (Targeted Attack) 与 随机失效 (Random Failure)]")
    print("[随机失效 (Random Failure)]：基于均匀概率分布的节点渗流过程。即基于均匀概率分布模拟自然故障或突发偶然事故；")
    print("           旨在模拟设备老化、供电故障等离散型突发事故，作为评估系统天然冗余度的性能基准 (Baseline)。")
    print("[蓄意攻击 (Targeted Attack)]：基于节点中心性序位（如介数 Betweenness 或 度数 Degree）的确定性移除策略，旨在模拟针对核心枢纽的精确破坏。")
    print("           用以识别网络结构的拓扑脆弱性 (Vulnerability)。")
    print("-" * 84 + "\n")

def simulate_robustness_pro(city_name, num_random_runs=20):
    """
    执行深度鲁棒性模拟，并自动打印包含名词解释、方案评估、异常现象说明的完整报告。
    """
    # 1. 初始数据预处理
    G_orig, _ = load_data(city_name)
    if G_orig is None: return
    
    # 提取最大连通分量作为分析基准
    lcc_nodes = max(nx.connected_components(G_orig), key=len)
    G = G_orig.subgraph(lcc_nodes).copy()
    N = G.number_of_nodes()
    initial_efficiency = nx.global_efficiency(G)
    
    # 设置移除步长 (1% 间隔)
    step = max(1, int(N * 0.01))
    removal_steps = list(range(0, N + 1, step)) 
    if removal_steps[-1] != N: removal_steps.append(N)

    def run_attack_sim(node_list):
        """内部仿真引擎：按顺序移除节点并记录拓扑指标"""
        s_list, e_list = [1.0], [initial_efficiency]
        temp_G = G.copy()
        current_removed = 0
        snapshot_data = {} # 新增：用于存储关键百分比点的数据

        for target_remove in removal_steps[1:]:
            while current_removed < target_remove and current_removed < len(node_list):
                node_to_del = node_list[current_removed]
                if temp_G.has_node(node_to_del):
                    temp_G.remove_node(node_to_del)
                current_removed += 1
            
            # 计算受损后的指标
            if temp_G.number_of_nodes() > 1:
                s_val = len(max(nx.connected_components(temp_G), key=len)) / N
                e_val = nx.global_efficiency(temp_G)
            else:
                s_val, e_val = 0, 0
            
            s_list.append(s_val); e_list.append(e_val)

            # --- 只增不减：捕捉特定百分比下的数据快照 ---
            current_pct = round(current_removed / N * 100)
            if current_pct in [5, 10, 25, 50, 75] and current_pct not in snapshot_data:
                snapshot_data[current_pct] = {'S': s_val, 'E': e_val}

        return np.array(s_list), np.array(e_list), snapshot_data

    # 2. 建立对比基准：随机失效模拟 (Monte Carlo 采样)
    print(f"--- {city_name} 城市地铁网络鲁棒性深度评估报告 ---")
    print(f"[数据执行] 正在执行 {num_random_runs} 轮随机失效实验以计算统计基准...")
    
    s_rand_all, e_rand_all = [], []
    random_snapshots_collection = [] # 用于收集多轮随机实验的快照

    for _ in range(num_random_runs):
        nodes_rand = list(G.nodes()); random.shuffle(nodes_rand)
        s, e, snaps = run_attack_sim(nodes_rand)
        s_rand_all.append(s); e_rand_all.append(e)
        random_snapshots_collection.append(snaps)
    
    s_random_mean, s_random_std = np.mean(s_rand_all, axis=0), np.std(s_rand_all, axis=0)
    e_random_mean, e_random_std = np.mean(e_rand_all, axis=0), np.std(e_rand_all, axis=0)

    # 计算随机失效在各阈值点的平均性能
    random_threshold_means = {}
    for pct in [5, 10, 25, 50, 75]:
        s_vals = [d[pct]['S'] for d in random_snapshots_collection if pct in d]
        e_vals = [d[pct]['E'] for d in random_snapshots_collection if pct in d]
        random_threshold_means[pct] = {'S': np.mean(s_vals) if s_vals else 0, 
                                        'E': np.mean(e_vals) if e_vals else 0}

    # 3. 策略性打击模拟：四大中心性指标对比
    for attack_metric in ['Betweenness', 'Degree', 'Closeness', 'Eigenvector']:
        # 计算节点重要性权重 (保持原样)
        if attack_metric == 'Betweenness': cent = nx.betweenness_centrality(G)
        elif attack_metric == 'Degree': cent = nx.degree_centrality(G)
        elif attack_metric == 'Closeness': cent = nx.closeness_centrality(G)
        else:
            try: cent = nx.eigenvector_centrality(G, max_iter=1000)
            except: cent = nx.degree_centrality(G)
        
        # 按权重降序生成攻击序列
        nodes_targeted = sorted(cent, key=cent.get, reverse=True)
        s_targeted, e_targeted, targeted_snaps = run_attack_sim(nodes_targeted)

        # 鲁棒性 R 指数计算
        r_s = np.trapz(s_targeted) / len(s_targeted)
        r_e = np.trapz(e_targeted) / len(e_targeted)

        # --- 打印该城市的实时分析报告 ---
        print(f"--- {city_name} 网络鲁棒性分析报告 ---")

        # --- 只增不减：新增多阶段数据 snapshot 报告 ---
        print(f"\n[攻击效能快照：基于 {attack_metric} 指标的 5%/10%/25%/50%/75% 节点移除对比]")
        print(f"{'移除比例':<10} | {'针对性S':<10} | {'随机均值S':<10} | {'针对性E':<10} | {'随机均值E':<10}")
        print("-" * 75)
        for pct in [5, 10, 25, 50, 75]:
            t_s = targeted_snaps.get(pct, {'S': 0})['S']
            t_e = targeted_snaps.get(pct, {'E': 0})['E']
            r_s_base = random_threshold_means.get(pct, {'S': 0})['S']
            r_e_base = random_threshold_means.get(pct, {'E': 0})['E']
            print(f"{pct}% 移除   | {t_s:.4f}     | {r_s_base:.4f}     | {t_e:.4f}     | {r_e_base:.4f}")

        print("\n[系统稳健性参数解析]")
        print(f"1. 结构鲁棒性指数 (R_s = {r_s:.3f})：反映移除基于 {attack_metric} 指标的关键站点时，物理连通性瓦解的速率。")
        print(f"2. 功能鲁棒性指数 (R_e = {r_e:.3f})：反映核心节点缺失后，线网整体通勤效率的衰减动态。")

        # 4. 统计图表可视化 (保持原样逻辑，修正负数阴影)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
        x_axis = np.array(removal_steps) / N * 100 
        
        # 绘制 S指标 崩塌曲线
        ax1.plot(x_axis, s_targeted, color='red', marker='o', label=f'Targeted Attack ({attack_metric})', markersize=3)
        ax1.plot(x_axis, s_random_mean, color='green', label='Random Failure(mean)')
        ax1.fill_between(x_axis, np.maximum(0, s_random_mean - s_random_std), s_random_mean + s_random_std, color='green', alpha=0.1)
        ax1.set_title(f"Structural Integrity (S)\nRobustness Index R: {r_s:.3f}"); ax1.set_ylim(0, 1.05)
        ax1.set_xlabel("Actual Nodes Removed (%)"); ax1.set_ylabel("Giant Component Ratio"); ax1.legend()

        # 绘制 E指标 衰减曲线
        ax2.plot(x_axis, e_targeted, color='red', marker='o', label=f'Targeted Attack ({attack_metric})', markersize=3)
        ax2.plot(x_axis, e_random_mean, color='green', label='Random Failure(mean)')
        ax2.fill_between(x_axis, np.maximum(0, e_random_mean - e_random_std), e_random_mean + e_random_std, color='green', alpha=0.1)
        ax2.set_title(f"Travel Efficiency (E)\nRobustness Index R: {r_e:.3f}"); ax2.set_ylim(0, initial_efficiency * 1.1)
        ax2.set_xlabel("Actual Nodes Removed (%)"); ax2.set_ylabel("Global Efficiency"); ax2.legend()
        
        plt.tight_layout()
        plt.show()
        print("-" * 85)


    # 5. 模型局限性与优势评估 (保持原样)
    print("\n[模型局限性与优势评估]")
    print("1. 核心优势：引入多轮 Monte Carlo 模拟建立随机对照组，有效剥离偶然性因素；采用 AUC 积分法实现了异质网络鲁棒性的标准化对比。")
    print("2. 学术局限：当前实验基于初始静态拓扑序位进行移除，尚未模拟网络崩塌过程中客流在剩余拓扑结构上的动态重分配行为（即非自适应攻击）。")
    print("\n[关键异常观测说明]")
    print("观测现象：当节点移除比例极高(>90%)时，E 曲线可能出现非物理性的数据回升。")
    print("物理机制：此为复杂网络计算中的‘小样本偏差’现象。当网络降解至仅剩极少数节点（如 2-3 个）且恰好保持直连时，")
    print("根据全局效率定义，这组极小样本间的路径极短，导致数学计算上的‘效率提升’伪影。")
    print("因此，当网络降解至仅剩极少数节点且恰好保持直连时，根据效率公式定义，这组极小样本间的路径极短，导致数学计算上的“效率提升”假象。")
    print("判定结论：该跳变不具交通规划意义。评估系统韧性时，应重点关注曲线前中期的斜率变化及崩溃阈值。")

# --- 交互界面启动 ---
city_options = get_city_list(DATA_PATH)
if city_options:
    print_terminology_report()
    widgets.interact(
        simulate_robustness_pro, 
        city_name=widgets.Dropdown(options=city_list, description='分析城市:'),
        num_random_runs=widgets.Dropdown(options=[10, 20, 50], value=20, description='随机模拟次数:')
    )

------------------------------ 专业名词与参数定义说明 ------------------------------
[1. 宏观拓扑完整性 (Size of the Largest Connected Component, S值)]
 [专业定义]：受损网络中最大连通子图（巨分量LCC）包含的节点数与初始网络总节点数的比值。
 [物理意义]：作为渗流理论中的序参数 (Order Parameter)，S值量化了地铁系统物理骨架的宏观连通水平。反映地铁系统的‘物理骨骼’是否维持全局连通。
           它表征了系统在遭受扰动后，仍能维持跨区域通达能力的骨干结构规模。
           当 S值 发生阶跃式下降、跌破临界点意味着网络发生相变时，标志着网络已进入彻底碎片化的结构相变点。

[2. 全局传输效能 (Global Efficiency, E值)]
[专业定义]：网络中所有节点对之间最短路径长度倒数的算术平均值。
[物理意义]：衡量系统功能的有效性。作为系统功能的有效性测度，反映了线网在遭受破坏过程中的性能降级 (Performance Degradation)。
           由于该指标考虑了路径长度的倒数，它能够有效评估网络在分裂为多个孤岛后，残余结构对于乘客通勤的平均通达效率。
           即使物理骨架（S值）尚存，若核心枢纽失效导致绕路开销剧增，E值仍会显著下降，揭示了网络服务水平的丧失。
           因此这反映乘客在网内通勤的通达便捷程度；若关键枢纽失效导致绕路开销剧增，E值将显著下降。

[3. 累积鲁棒性指数 (Robustness Index, R值)]
[专业定义]：性能衰减曲线（S或E指标）在节点移除全过程中的累积积分面积，即与坐标轴围成的积分面积(Area Under Curve, AUC)。
 [物理意义]: 量化系统在遭受‘连续性打击’全过程中的平均生存效能。R值将瞬时的崩溃转折与长期的衰减过程整合为一个标量,数值范围 [0, 0.5]。
           数值越大代表系统韧性越优，即数值越大代表了系统在连续移除压力下表现出的拓扑稳健性与抗打击能力越优。

[4. 蓄意攻击 (Targeted Attack) 与 随机失效 (Random Failure)]
[

interactive(children=(Dropdown(description='分析城市:', options=('Beijing', 'Changchun', 'Changsha', 'Changzhou', …

In [61]:
# ================= 代码段 3: 自适应鲁棒性全维度分析 (学术深度增强版) =================

import random
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import display, HTML
import ipywidgets as widgets

# --- 1. 全局学术名词与高级参数定义报告 ---
def print_comprehensive_theory_report():
    print("-" * 30 + " 复杂网络鲁棒性理论参数词典 " + "-" * 30)

    print("[1. 宏观结构连通度 (Size of Giant Component, S)]")
    print("[专业定义]：受损网络中最大连通子图（LCC）包含的节点数与初始网络规模的比值。")
    print("[物理意义]：作为渗流理论中的序参数 (Order Parameter)，它直接反映了地铁系统的“物理骨架”是否维持了全局范围内的空间可达性。")
    print("          衡量网络中依然保持互联的最大碎片占原始网络规模的比例。它是反映地铁网‘物理完整性’的核心指标。")

    print("\n[2. 全局传输效能 (Global Efficiency, E)]")
    print("[专业定义]：网络中所有节点对之间最短路径长度倒数的算术平均值。")
    print("[物理意义]：量化系统的“服务效能”。即使物理结构尚未彻底碎裂，若核心枢纽失效导致绕路开销剧增，E值亦会迅速崩塌，揭示了网络通勤质量的丧失。")
    print("          计算所有车站对之间通行效率的平均值。它捕捉的是‘通勤成本’，即使网没断，绕路严重也会导致E值剧降。")

    print("\n[3. 临界崩溃阈值 (Critical Threshold, fc)]")
    print("[专业定义]：在本实验中，将 S < 0.1 (即网络最大连通子图已不足原始规模的 10%) 设定为‘崩溃阈值’。S值首次跌破 0.1 (10%) 时对应的节点移除比例。")
    print("[设定依据]：参照渗流理论(Percolation Theory)，当系统失去90%的通达能力时，其作为社会基础设施的功能已完全失效。")
    print("   选择 0.1 的讲究：这是一种标准化标尺。不同城市车站数不同，通过统一百分比，可公平对比不同规模城市的耐受极限。")

    print("\n[4. 自适应攻击逻辑 (Adaptive Attack Strategy)]")
    print("[算法定义]：在每一个节点的移除步骤后，实时重新计算剩余网络的中心性排名，并据此决定下一个攻击目标。")
    print("   学术价值：模拟具备实时动态情报的极端破坏行为。相较于传统的静态攻击，该模式能更真实地揭示网络拓扑重构过程中的连锁失效风险。")
    print("-" * 85 + "\n")

# --- 2. 核心模拟函数 ---
def simulate_robustness_adaptive_pro(city_name, num_random_runs=20):
    """
    执行深度鲁棒性模拟，并自动输出包含名词解释、BCDE策略分析、全阈值快照对比、方案评估的完整学术报告。
    """
    # 加载数据
    G_orig, _ = load_data(city_name)
    if G_orig is None: return
    lcc_nodes = max(nx.connected_components(G_orig), key=len)
    G = G_orig.subgraph(lcc_nodes).copy()
    N = G.number_of_nodes()
    initial_eff = nx.global_efficiency(G)
    
    metrics = ['Degree', 'Betweenness', 'Closeness', 'Eigenvector']

    def get_centrality(graph, method):
        if method == 'Betweenness': return nx.betweenness_centrality(graph)
        if method == 'Degree': return nx.degree_centrality(graph)
        if method == 'Closeness': return nx.closeness_centrality(graph)
        if method == 'Eigenvector':
            try: return nx.eigenvector_centrality(graph, max_iter=500)
            except: return nx.degree_centrality(graph)
        return nx.degree_centrality(graph)

    # 定义 1% 步长的采样点
    step_size = max(1, int(N * 0.01))
    removal_points = list(range(0, N + 1, step_size))
    if removal_points[-1] != N: removal_points.append(N)
    x_axis_pct = np.array(removal_points) / N * 100

    def run_sim(mode, metric_name=None, static_nodes=None):
        temp_G = G.copy()
        s_list, e_list = [1.0], [initial_eff]
        current_removed = 0
        snapshot_data = {} # 记录特定阈值点的数据

        for target_remove in removal_points[1:]:
            num_to_rem = target_remove - current_removed
            if mode == 'Adaptive' and temp_G.number_of_nodes() > 0:
                curr_cent = get_centrality(temp_G, metric_name)
                target_nodes = sorted(curr_cent, key=curr_cent.get, reverse=True)
            elif mode == 'Static':
                target_nodes = [n for n in static_nodes if n in temp_G]
            else: # Random Failure
                target_nodes = list(temp_G.nodes()); random.shuffle(target_nodes)

            nodes_to_del = target_nodes[:num_to_rem]
            temp_G.remove_nodes_from(nodes_to_del)
            current_removed += len(nodes_to_del)

            if temp_G.number_of_nodes() > 1:
                l_cc = len(max(nx.connected_components(temp_G), key=len)) / N
                eff = nx.global_efficiency(temp_G)
            else:
                l_cc, eff = 0, 0
            
            s_list.append(l_cc); e_list.append(eff)

            # 捕捉关键百分比点用于报告输出
            pct_current = round(current_removed / N * 100)
            if pct_current in [5, 10, 25, 50, 75] and pct_current not in snapshot_data:
                snapshot_data[pct_current] = (l_cc, eff)

        return np.array(s_list), np.array(e_list), snapshot_data

    # --- 阶段 A：输出分析方案综合评估 ---
    print(f"--- {city_name} 网络自适应鲁棒性压力测试报告 ---")
    print("\n[当前实验方案评估]")
    print("简单评估")
    print("1. 动态还原：采用自适应重算逻辑，精准捕获了网络受损过程中重要性迁移的动态特征。")
    print("2. 统计严谨性：通过多轮蒙特卡洛 (Monte Carlo) 采样建立随机失效基准带宽（绿色阴影区），确保了实验结果具有统计学意义上的显著性。")
    print("3. 物理边界修正：通过 np.maximum 逻辑截断绘图边界，消除了统计波动产生的负值伪影。")
    print("\n[方案学术局限]")
    print("1. 算力开销：自适应重算 (特别是介数指标) 具有极高的计算复杂度，对超大型地铁线网分析耗时相对较长。")
    print("2. 静态客流限制：当前模型基于拓扑物理属性，暂未融合动态实时客流负荷的溢出效应。")
    print("  即模型主要聚焦于拓扑结构韧性，暂未完全耦合乘客在路径阻断后的动态换乘选择及客流重分配 (Traffic Redistribution) 效应。")

    # --- 阶段 B：建立随机失效统计基准 ---
    print(f"\n[状态] 正在执行 {num_random_runs} 轮随机失效模拟，计算统计基准...")
    s_rand_all, e_rand_all = [], []
    random_snapshots_collection = []
    for _ in range(num_random_runs):
        sr, er, snap = run_sim('Random')
        s_rand_all.append(sr); e_rand_all.append(er)
        random_snapshots_collection.append(snap)
    
    s_rand_mean, s_rand_std = np.mean(s_rand_all, axis=0), np.std(s_rand_all, axis=0)
    e_rand_mean, e_rand_std = np.mean(e_rand_all, axis=0), np.std(e_rand_all, axis=0)

    # 预处理随机失效在各阈值点的平均均值
    random_base_table = {}
    for pct in [5, 10, 25, 50, 75]:
        s_vals = [d[pct][0] for d in random_snapshots_collection if pct in d]
        e_vals = [d[pct][1] for d in random_snapshots_collection if pct in d]
        random_base_table[pct] = (np.mean(s_vals), np.mean(e_vals))

    summary_list = []

    # --- 阶段 C：遍历 BCDE 维度进行蓄意攻击与专项报告 ---
    for metric in metrics:
        print(f"\n{'-' * 10} 执行策略深度分析：基于 {metric} 的蓄意攻击 {'-' * 10}")
        
        # 对应指标的学术策略解析
        if metric == 'Degree':
            print("[策略导向]：连接广度打击。移除物理连接线最多的换乘大站，旨在瘫痪局部网格的聚合性。")
        elif metric == 'Betweenness':
            print("[策略导向]：枢纽咽喉打击。移除承载路径最多的桥梁站点，旨在切断跨区域的长距离通达。")
        elif metric == 'Closeness':
            print("[策略导向]：逻辑中心打击。移除拓扑距离各处最近的枢纽，旨在最大化增加乘客绕路开销。")
        else:
            print("[策略导向]：影响力朋友圈打击。移除连接了其他重要核心的地位显赫站点。")

        static_cent = get_centrality(G, metric)
        nodes_static = sorted(static_cent, key=static_cent.get, reverse=True)
        
        # 执行自适应与静态对比实验
        s_adaptive, e_adaptive, adaptive_snap = run_sim('Adaptive', metric)
        s_static, e_static, _ = run_sim('Static', static_nodes=nodes_static)
        
        # 识别自适应 fc (S < 0.1)
        fc_val = 100.0
        for idx, val in enumerate(s_adaptive):
            if val < 0.1: fc_val = x_axis_pct[idx]; break

        # --- 输出全阈值性能快照对比表 ---
        print(f"\n[{metric} 自适应攻击多阶段性能快照对比]")
        print(f"{'移除比例':<10} | {'自适应S':<10} | {'随机失效S':<10} | {'自适应E':<10} | {'随机失效E':<10}")
        print("-" * 75)
        for pct in [5, 10, 25, 50, 75]:
            as_val, ae_val = adaptive_snap.get(pct, (0, 0))
            rs_base, re_base = random_base_table.get(pct, (0, 0))
            print(f"{pct}% 移除   | {as_val:.4f}     | {rs_base:.4f}     | {ae_val:.4f}     | {re_base:.4f}")

        # 4. 可视化绘图 (物理边界修正)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
        
        # S图：连通性崩塌
        ax1.plot(x_axis_pct, s_adaptive, color='red', marker='o', label=f'Adaptive {metric}', markersize=3)
        ax1.plot(x_axis_pct, s_static, color='orange', linestyle='--', label='Static Attack', alpha=0.7)
        ax1.plot(x_axis_pct, s_rand_mean, color='green', label='Random Baseline')
        ax1.fill_between(x_axis_pct, np.maximum(0, s_rand_mean - s_rand_std), 
                         s_rand_mean + s_rand_std, color='green', alpha=0.1)
        ax1.axvline(x=fc_val, color='red', linestyle=':', label=f'Adaptive fc: {fc_val:.1f}%')
        ax1.set_title(f"Connectivity Decay (S)\nAdaptive fc: {fc_val:.1f}%", fontweight='bold')
        ax1.set_ylim(0, 1.05); ax1.set_xlabel("Actual Nodes Removed (%)"); ax1.legend(frameon=False)

        # E图：效率衰减
        ax2.plot(x_axis_pct, e_adaptive, color='blue', marker='o', label='Adaptive Efficiency', markersize=3)
        ax2.plot(x_axis_pct, e_rand_mean, color='green', label='Random Efficiency')
        ax2.fill_between(x_axis_pct, np.maximum(0, e_rand_mean - e_rand_std), 
                         e_rand_mean + e_rand_std, color='green', alpha=0.1)
        ax2.set_title(f"Efficiency Decay (E)\nInitial E: {initial_eff:.4f}", fontweight='bold')
        ax2.set_ylim(0, initial_eff * 1.1); ax2.set_xlabel("Actual Nodes Removed (%)"); ax2.legend(frameon=False)
        
        plt.suptitle(f"{city_name} 自适应鲁棒性压力测试: {metric}", fontsize=18, y=1.05)
        plt.tight_layout(); plt.show()
        print("-" * 92)

        summary_list.append({
            "评估指标": metric, "自适应崩塌点 (fc)": f"{fc_val:.2f}%",
            "自适应鲁棒指数 (R)": f"{np.trapz(s_adaptive)/len(s_adaptive):.4f}",
            "静态鲁棒指数 (R)": f"{np.trapz(s_static)/len(s_static):.4f}"
        })

    # --- 阶段 D：实验现象与数据总结 ---
    print("\n" + "="*30 + " 实验观测现象学术说明 " + "="*30)
    print("[1. 关于结构连通度(S)与出行效率(E)衰减速率的差异]")
    print("   观察发现：通常 E指标 的下降速度显著快于 S指标。")
    print("   物理机制：这说明枢纽节点的失效对线网‘服务效能’的破坏具有超前性。即网络尚未发生大面积碎裂前，")
    print("   由于关键换乘路径的阻断，乘客的平均通勤代价已经发生了剧烈跳升。")
    print("\n[2. 关于曲线末端‘异常数据反弹’的定性说明]")
    print("   原因：此为典型‘极小样本偏差’产生的计算伪影。在移除超过 90% 节点后，网络已处于崩溃边缘。")
    print("   若残存的极少数站点恰好相连，分母效应会导致数学上的效率瞬时冲高，其不具备实际交通决策参考价值。")
    print("\n[3. 静态 vs 自适应攻击的“分化点”]")
    print("现象说明：红色实线（自适应）通常位于橙色虚线（静态）下方。")
    print("物理判定：这揭示了网络中存在大量“隐性核心”，当显性枢纽被拔除后，这些节点会迅速接替成为新的压力点。")
    print("自适应攻击由于能实时定位这些迁徙的核心，其破坏效率显著更高。")
    print(f"\n--- {city_name} 网络自适应鲁棒性量化分析汇总表 ---")
    display(pd.DataFrame(summary_list))

# --- 交互界面部署 ---
city_list = get_city_list(DATA_PATH)
if city_list:
    print_comprehensive_theory_report()
    widgets.interact(
        simulate_robustness_adaptive_pro, 
        city_name=widgets.Dropdown(options=city_list, description='分析城市:'),
        num_random_runs=widgets.Dropdown(options=[10, 20, 50], value=20, description='模拟次数:')
    )

------------------------------ 复杂网络鲁棒性理论参数词典 ------------------------------
[1. 宏观结构连通度 (Size of Giant Component, S)]
[专业定义]：受损网络中最大连通子图（LCC）包含的节点数与初始网络规模的比值。
[物理意义]：作为渗流理论中的序参数 (Order Parameter)，它直接反映了地铁系统的“物理骨架”是否维持了全局范围内的空间可达性。
          衡量网络中依然保持互联的最大碎片占原始网络规模的比例。它是反映地铁网‘物理完整性’的核心指标。

[2. 全局传输效能 (Global Efficiency, E)]
[专业定义]：网络中所有节点对之间最短路径长度倒数的算术平均值。
[物理意义]：量化系统的“服务效能”。即使物理结构尚未彻底碎裂，若核心枢纽失效导致绕路开销剧增，E值亦会迅速崩塌，揭示了网络通勤质量的丧失。
          计算所有车站对之间通行效率的平均值。它捕捉的是‘通勤成本’，即使网没断，绕路严重也会导致E值剧降。

[3. 临界崩溃阈值 (Critical Threshold, fc)]
[专业定义]：在本实验中，将 S < 0.1 (即网络最大连通子图已不足原始规模的 10%) 设定为‘崩溃阈值’。S值首次跌破 0.1 (10%) 时对应的节点移除比例。
[设定依据]：参照渗流理论(Percolation Theory)，当系统失去90%的通达能力时，其作为社会基础设施的功能已完全失效。
   选择 0.1 的讲究：这是一种标准化标尺。不同城市车站数不同，通过统一百分比，可公平对比不同规模城市的耐受极限。

[4. 自适应攻击逻辑 (Adaptive Attack Strategy)]
[算法定义]：在每一个节点的移除步骤后，实时重新计算剩余网络的中心性排名，并据此决定下一个攻击目标。
   学术价值：模拟具备实时动态情报的极端破坏行为。相较于传统的静态攻击，该模式能更真实地揭示网络拓扑重构过程中的连锁失效风险。
-------------------------------------------------------------------------------------



interactive(children=(Dropdown(description='分析城市:', options=('Beijing', 'Changchun', 'Changsha', 'Changzhou', …